# 학습 관제

**실행은 커널 밖에서, 보기만 여기서.**
Jupyter 커널은 브라우저가 끊기면 죽는다 (이슈 S15P21A103-111 실측).
30분짜리 학습을 셀에서 직접 돌리면 같이 죽으므로 `launch()` 로 떼어낸다.

로직은 `AI/tracking/monitor.py` 에 있다. 노트북은 얇게 유지한다 —
분석 코드가 셀에만 있으면 재현도 리뷰도 안 된다.

In [ ]:
import sys
from pathlib import Path

AI = Path.home() / "S15P21A103" / "AI"
sys.path.insert(0, str(AI))

from tracking import monitor as m
print(m.AI_ROOT, m.EXP_LOG.exists())

## 1. 잡 띄우기

커널을 닫아도 산다. 로그는 `AI/out/<이름>`.

In [ ]:
# m.launch(
#     "python tools/collect_sim.py --episodes 100 --jitter 0.05 --seed 0 "
#     "--skill-id pick_place --out datasets/sim_pick_v6 --log",
#     "collect_v6.log")

# m.launch(
#     "python tools/repeat_runs.py --data datasets/sim_pick_v6 --runs 3 "
#     "--episodes 100 --seed-base 0 --eval-seed-base 3000 --device cuda --log",
#     "repeat_v6.log")

## 2. 진행 상황

반복 실행해도 된다. 체크포인트가 하나씩 늘면 도는 중이다.

In [ ]:
m.progress("repeat_v5.log", pattern="sim_pick_v5*.pt")

## 3. 실험 이력

**평균이 아니라 `ci95%` 열을 봐라.** 구간이 겹치는 두 실행은 이 n 으로 구분되지 않는다.

그리고 `val_loss` 와 `mean%` 를 나란히 두고 봐라 —
이 프로젝트에서 둘의 상관은 이미 두 번 깨졌다
(v3 0.053→4.3% / v4 0.058→0.0%).

In [ ]:
m.runs_table()

## 4. 실패 모양

성공률은 "얼마나 못 하나"만 말한다. **왜** 못 하는지는 이쪽이다.

읽는 법 — 실측 재생 허용오차는 **±10mm 에서 3/4** 다 🟢
- `닫는거리mm` 가 10 을 넘으면 손실이 어떻든 성공할 수 없다 (scripted 는 4.7)
- `접촉%` 가 0 에 가까우면 물체까지 못 간 것이다
- `최근접3d_mm` 이 크면 xy 는 맞췄어도 **높이가 안 내려간** 것이다

In [ ]:
m.failure_shape(limit=6)